# Task 6: RAG Pipeline (Generator)

**Goal:** Complete baseline RAG with LLM generation

**Components:**
- Vector Store: Chroma (from Task 4-5)
- LLM: gpt-4o-mini
- Prompt: Context-grounded QA with source citation

## 1. Setup

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found"

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

CHROMA_DIR = Path("../data/chroma_db")

print("Setup complete ✅")

Setup complete ✅


## 2. Load Vector Store & Create Retriever

In [2]:
# Load existing vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=embeddings,
    collection_name="lg_manuals",
)

print(f"Loaded vector store: {vectorstore._collection.count()} documents")

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print("Retriever created (k=5) ✅")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Loaded vector store: 320 documents
Retriever created (k=5) ✅


## 3. Setup LLM & Prompt

In [3]:
# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

# RAG Prompt (from WEEK3_TASKS.md §3.3)
RAG_PROMPT = """당신은 LG 가전제품 사용설명서를 안내하는 QA Assistant입니다.

아래 제공된 context만 사용해서 질문에 답변하세요.
context에 없는 내용은 추측하지 말고 "문서에서 확인할 수 없습니다"라고 답변하세요.
답변 마지막에는 참고한 매뉴얼의 모델명과 페이지 번호를 함께 적어주세요.

[Context]
{context}

[Question]
{question}

[Answer]"""

prompt = ChatPromptTemplate.from_template(RAG_PROMPT)
print("LLM and prompt ready ✅")

LLM and prompt ready ✅


## 4. Build RAG Chain

In [4]:
def format_docs(docs):
    """Format retrieved documents for context."""
    formatted = []
    for doc in docs:
        meta = doc.metadata
        source_info = f"[{meta.get('model_name', 'unknown')} p.{meta.get('page', '?')}]"
        formatted.append(f"{source_info}\n{doc.page_content}")
    return "\n\n---\n\n".join(formatted)

# RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built ✅")

RAG chain built ✅


## 5. Test RAG Pipeline

In [5]:
def ask(question: str):
    """Ask a question and display the answer."""
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    
    answer = rag_chain.invoke(question)
    print(f"\nA: {answer}")
    
    return answer

In [6]:
# Test queries
test_queries = [
    "정수기 필터 교체는 어떻게 하나요?",
    "공기청정기 필터 청소 방법을 알려주세요.",
    "청소기 배터리 충전 시간은 얼마나 되나요?",
    "Wi-Fi 연결이 안 될 때 어떻게 하나요?",
]

In [7]:
# Run all test queries
answers = []
for q in test_queries:
    ans = ask(q)
    answers.append({"question": q, "answer": ans})


Q: 정수기 필터 교체는 어떻게 하나요?


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



A: 정수기 필터 교체 방법은 다음과 같습니다:

1. 필터 커버의 아래 부분을 잡고 앞으로 당깁니다.
2. 필터 끝부분을 들어올린 후 반시계 방향으로 돌려 분리합니다. 이때 내부 압력에 의해 약간의 물이 떨어질 수 있으니 아래에 마른 수건을 놓아두세요.
3. 새 필터의 라벨이 정면으로 향하도록 시계 방향(오른쪽)으로 더 이상 돌아가지 않을 때까지 돌려 조립합니다.
4. 필터 체결부를 솔로 청소합니다.
5. 필터가 완전히 장착되었는지 확인하고, 두 손으로 필터 커버를 닫습니다. 왼쪽 상단부터 조립하며, 닫으면 자동으로 필터 세척이 약 7분간 진행됩니다.

필터 교체 주기는 중금속9 흡착 필터는 6개월, 바이러스 클리어 필터는 12개월입니다. 필터 교체 후에는 필터 사용량을 초기화해야 합니다.

참고한 매뉴얼: WD523A**, WD524A**, WD323A**, WD520A**, WD507A**, WD508A** (p.28)

Q: 공기청정기 필터 청소 방법을 알려주세요.

A: 공기청정기 필터 청소 방법은 다음과 같습니다:

1. 제품의 커버를 분리하세요.
2. 진공 청소기 또는 부드러운 솔을 이용해 토탈 알러지 집진 필터를 감싸고 있는 극세필터의 먼지를 제거하세요.
3. 청소가 끝난 후에는 커버를 다시 장착하세요.

필터 청소 후에는 상태 표시부의 필터 교체 알림이 표시될 수 있으니, 필요 시 필터 교체 알림을 해제하는 방법도 참고하세요.

참고한 매뉴얼: airpurifier_simple, p.37

Q: 청소기 배터리 충전 시간은 얼마나 되나요?

A: 문서에서 확인할 수 없습니다. (모델명: MFL69726859, 페이지 번호: 28)

Q: Wi-Fi 연결이 안 될 때 어떻게 하나요?

A: Wi-Fi 연결이 안 될 때는 다음과 같은 조치를 취해보세요:

1. 가전제품 전원 플러그를 뽑고 약 1분 뒤 다시 진행해 보세요.
2. 무선 공유기에 방화벽이 설정되어 있다면 예외를 설정하거나 가정에서 사용 중인 Wi-Fi를 스마트폰에 등록 또는 해제 후 다

## 6. Evaluation Checklist

For each answer, check:

| Criteria | Description |
|----------|-------------|
| **Grounded** | Answer based on context, not hallucinated? |
| **Complete** | Answers the question fully? |
| **Source cited** | Includes model name and page number? |
| **Correct product** | References the right product category? |

In [8]:
# Quick view of retrieved context for debugging
def debug_retrieval(question: str):
    """Show what was retrieved for a question."""
    docs = retriever.invoke(question)
    print(f"\nRetrieved {len(docs)} docs for: {question}")
    print("-" * 40)
    for i, doc in enumerate(docs):
        meta = doc.metadata
        print(f"[{i+1}] {meta.get('model_name')} p.{meta.get('page')} | {meta.get('category')}")
    return docs

In [9]:
# Debug first query
debug_retrieval(test_queries[0])


Retrieved 5 docs for: 정수기 필터 교체는 어떻게 하나요?
----------------------------------------
[1] waterpurifier_complex p.28 | waterpurifier
[2] MFL69726859 p.37 | airpurifier
[3] MFL71817002 p.20 | waterpurifier
[4] airpurifier_simple p.37 | airpurifier
[5] MFL71817002 p.19 | waterpurifier


[Document(id='df669454-a094-4ea8-bbde-821ad3562020', metadata={'category': 'waterpurifier', 'char_count': 779, 'chunk_id': 'waterpurifier_complex_p028_c000', 'chunk_index': 0, 'complexity': 'complex', 'model_name': 'waterpurifier_complex', 'page': 28, 'source': 'waterpurifier_complex.pdf'}, page_content='28 관리하기\n• 정수 필터를 분리할 때 내부 압력에 의해 약간의 물이\n알아두기\n떨어질 수 있습니다. 아래에 마른 수건을 놓으세요.\n물이 얼 때 물속의 미네랄이 중심축으로 모여 서로\n결합하여 대부분 탄산칼슘으로 변하고 얼음 속에서\n하얗게 보여요. 또한 얼음이 녹으면, 탄산칼슘만 남아\n흰색 결정체가 보일 수 있어요.\n정수 필터 교체하기\n모델명: WD523A**, WD524A**, WD323A**,\nWD520A**, WD507A**, WD508A**\n주기적으로 필터 교체를 하지 않으면 정수기의 수질이\n3\n낮아질 수 있습니다. 솔을 이용하여 필터 체결부를 청소하세요.\n• 원수 수질, 물 사용량과 직수관/출수구 살균 기능\n사용량에 따라 예상 교체 주기가 달라질 수 있습니다.\n• 각 필터의 교체 주기는 다음과 같습니다.\n- 중금속9 흡착 필터: 6개월 (10 L/일 사용기준)\n- 바이러스 클리어 필터: 12개월 (10 L/일 사용기준)\n• 필터 교체 주기는 4인 가정 하루 10 L 사용을 기준으로\n하며, 필터의 수명은 수질, 수압, 계절, 지역에 따라\n차이가 있을 수 있습니다.\n• 상세한 필터 교체 방법이 필요하면 LG전자 홈페이지를\n참고하세요.\n1\n필터 커버의 아래 부분을 잡고 앞으로 당기세요.\n4\n새 필터의 라벨이 정면으로 향하도록 시계\n방향(오른쪽)으로 더 이상 돌아가지 않을 

## 7. Notes

### RAG Response Evaluation

| Query | Grounded | Complete | Source Cited | Correct Product |
|-------|----------|----------|--------------|-----------------|
| 정수기 필터 교체 | ✅ | ✅ | ✅ | ✅ |
| 공기청정기 필터 청소 | ✅ | ✅ | ✅ | ✅ |
| 청소기 배터리 충전 | ⚠️ | ❌ | ❌ WRONG | ❌ |
| Wi-Fi 연결 | ✅ | ✅ | ✅ | N/A |

**Score: 3/4 queries answered correctly**

### Observations

**Good:**
- Q1: LLM correctly focused on waterpurifier content despite airpurifier in context
- Q2: Accurate, grounded response with correct source
- Q4: Comprehensive Wi-Fi troubleshooting (cross-category OK)

**Bad:**
- Q3: Failed to answer AND cited wrong source (airpurifier for vacuum question)

### Baseline Limitations

**LIM-007: Wrong Source Citation on Failure**
- When LLM says "문서에서 확인할 수 없습니다", it still cites a source
- Cited source is WRONG product category
- Prompt needs improvement: don't cite source if can't answer

**LIM-008: Retrieval Gap for Specific Facts**
- Battery charging time query failed
- Either info not in chunks, or retrieval missed relevant docs
- Need to verify if this info exists in source PDFs

### Week 4 Improvement Candidates
- Fix prompt to not cite source on failure
- Metadata filtering to ensure correct product category
- Verify chunk coverage for common questions